# Exploratory Data Analysis (EDA)
This notebook explores the clinical patient records and ECG wave data.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import load_or_create_datasets
from src.utils.config import Config

config = Config(str(PROJECT_ROOT / "configs/params.yaml"), str(PROJECT_ROOT / "configs/paths.yaml"))
sns.set_theme(style="whitegrid")

In [ ]:
df, ecg = load_or_create_datasets(config)
print(f"Tabular shape: {df.shape}")
print(f"ECG shape: {ecg.shape}")
display(df.head())

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique()
})
display(summary)
display(df.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="cvd_target", ax=axes[0], color="#2f6f73")
axes[0].set_title("Target Distribution")
axes[0].set_xlabel("CVD target (0 = no, 1 = yes)")
axes[0].set_ylabel("Patients")

numeric_cols = df.select_dtypes(include=np.number).columns.drop("cvd_target")
corr = df[numeric_cols.tolist() + ["cvd_target"]].corr()[["cvd_target"]].sort_values("cvd_target")
sns.heatmap(corr, annot=True, cmap="vlag", center=0, ax=axes[1], cbar=False)
axes[1].set_title("Feature Correlation With Target")
plt.tight_layout()
plt.show()

In [ ]:
sample_idx = 0
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ecg[sample_idx, 0], color="#243b53", linewidth=0.8)
ax.set_title(f"Example ECG Waveform (patient index {sample_idx}, lead 1)")
ax.set_xlabel("Time sample")
ax.set_ylabel("Voltage")
plt.tight_layout()
plt.show()

In [ ]:
target_rates = df.groupby("smoking", dropna=False)["cvd_target"].mean().rename("cvd_rate")
display(target_rates.to_frame())

feature_cols = [column for column in df.columns if column not in {"patient_id", "cvd_target"}]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, column in zip(axes.flat, feature_cols[:6]):
    sns.boxplot(data=df, x="cvd_target", y=column, ax=axis, color="#8fcaca")
    axis.set_title(column.replace("_", " ").title())
plt.tight_layout()
plt.show()